## this contains the implementation of various excercises questions for video 1

### common imports & global values

In [1]:
from dataclasses import dataclass,field
import numpy as np
import torch
import torch.nn.functional as F
from itertools import islice
from random import shuffle

In [2]:

@dataclass
class ModelConfigs:
    """This class contains certain hyperparameter values that can be experimented upoin
    
        Class Members
            LEARNING_RATE: int - the step size taken during the optimization in gradient based approach
            REGURLARIZATION_VALUE: float - the loss factor on applied explicitly on the weights (L1/L2) regularization
            SMOOTHING_COUNT: int - make the counts array (counting approach) away from zero so the logs make more sense (away from inf probabilities)
            BOUNDARY_CHAR: str - start and end of the sequence
            SEED_VAL: int - pseudo random generator seed for reproducible samples.
    """
    LEARNING_RATE:int = field(default=-50)
    REGULARIZATION_VALUE:float = field(default=0.01)
    SMOOTHING_COUNT: int = field(default=1)
    START_BOUNDARY_CHAR: str = field(default=".")
    END_BOUNDARY_CHAR: str = field(default=".")
    SEED_VAL: int = field(default=2147483647)
    NUM_EPOCHS: int = field(default= 1000)
    DEV_SPLIT: float = field(default=.8)
    TEST_EVAL_SPLIT: float = field(default=.9)

In [3]:
@dataclass
class Vocab:
    """Vocab is a packaging class which contains important information for language modelling tasks

        Members:
            boundary_char: str - this signifies the start and end of a sequence
            vocab_letters: list[str] - this signifies the list of unique characters that occur in the data set
            stoi: dict[str, int] - this is the mapping between the the unique characters and int values
            itos: dict[int, str] -  this is the reverse of stoi
            n_unique: int - this is the number of unique characters + the boundary character
    """
    start_boundary_char: str
    end_boundary_char: str
    vocab_letters: list[str] = field(default_factory=list)
    stoi: dict[str, int] = field(default_factory=dict)
    itos: dict[int, str] = field(default_factory=dict)
    n_unique: int = field(default=0)


In [4]:
@dataclass
class NGram:
    """Packing class that contains the information related to the NGram classes

        Class Members
            data: list[tuple[str, ...]] - pairs of ngrams identified from within the sequence of strings
            n: length of the example subset (n = 2) means bigram, (n = 3) means trigram and so on.
    """
    data: list[tuple[str, ...]] = field(default_factory=list)
    n: int = field(default=2)

In [5]:
@dataclass
class Dataset:
    """Dataset class for holding the context and their leadning targets

        Class Member:
            X: torch.Tensor = the context vector one hot encoded
            Y: torch.Tensor = the target character for the corresponding context
    """
    X: torch.Tensor
    Y: torch.Tensor

In [6]:
def encode_context(ctx: torch.tensor, num_classes: int)->torch.tensor:
    """One hot encode the input context vector
        Args:
            ctx: context vector for the bigram model to be encoded
        returns:
            flattened tensor to be passed as input to the model
    """

    # from batch_size, context_size, num_chars -> batch_size, context_size * num_chars
    return F.one_hot(ctx, num_classes=num_classes).view(len(ctx), -1).float()

In [7]:
def get_names(data_path: str = '../names.txt')->list[str]:
    """Read the names file and load the contentents into memory as a list"""
    with open(data_path, 'r') as file:
        names =  file.read().splitlines()
        # names = list(set(names))
    return names

In [8]:
def create_vocab(start_boundary_char: str, end_boundary_char: str, data_set: list[str])->Vocab:
  """Creates a vocab object for the provided boundary character and list of strings


    Args:
      start_boundary_char: str - the start of a sequence
      end_boundary_char : str - the end of a sequence
      data_set: list[str] - list of character sequences that are to be modelled
    
    Returns
      vocab: Vocab - this is the vocab object that contains important character level model information.
  """


  # create a list of unique occuring characters from the set
  unique_chars = sorted(list(set(''.join(data_set))))
  
  # add boundary char at index 0
  unique_chars.insert(0, start_boundary_char)

  # create the string to int mapping for the unqiue characters
  stoi = {
    s: i
    for i, s in enumerate(unique_chars)
  }

  # reverse the above mapping to have int -> string
  itos = {
    i : s
    for s, i in stoi.items()
  }


  # package all together as a object to be used in other parts of the program
  return Vocab(
    start_boundary_char = start_boundary_char,
    end_boundary_char = end_boundary_char,
    vocab_letters= unique_chars[1:],
    stoi = stoi,
    itos=itos,
    n_unique=len(unique_chars)
  )
  



In [9]:
def generate_n_grams(data_set: list[str], start_boundary_char: str, end_boundary_char: str,n: int = 2)->NGram:
    """Function to generate the n-gram set of the input names dataset by default it generates bigram examples

    Args:
        data_set: list[str] = this is a list of strings (names)
        start_boundary_char : str = this is appended at the start of the string
        end_boundary_char: str = this is the appended at the end of the string

        n: int = if 2 we generate bigrams, 3 we generate tri-grams etc.

    Returns:
        n_grams = list[tupe[str,...]] -> this is a list of tuple each tuple will atleast have two elements (the input character and the resultant character) in case of bigrams
    """

    # resultant list
    n_grams = []

    for name in data_set:
        
        # append the boundary char
        char_list = [start_boundary_char] + list(name) + [end_boundary_char]

        for i in range(len(char_list) - n + 1):
            n_grams.append(tuple(char_list[i : i + n]))
    return NGram(
        n = n,
        data=n_grams
    )


In [10]:
def build_dataset(data_set: list[str], vocab: Vocab, n: int = 2)->Dataset:
    """For the given data split generate a data set object (context and target tensors)
        Args:
            data_set: list[str] = the train , eval or test split of the complete data set
            vocab: Vocab = the vocabulary created from the input data set
        Returns
            Dataset object which contains the encoded context vectors and targets for the corresponding contexts.
    """
    # create the n-gram examples for the data split
    n_gram = generate_n_grams(data_set=data_set, start_boundary_char=vocab.start_boundary_char, end_boundary_char=vocab.end_boundary_char, n = n)

    # store the context and their correspoding targets
    xs, ys = [], []
    # iterate over all the n_grams
    for data in n_gram.data:
        # create the context vector
        context = [vocab.stoi[ch] for ch in data[:-1]]
        # get the target for the example
        target = vocab.stoi[data[-1]]
        # add to the lists
        xs.append(context)
        ys.append(target)
    # convert the lists to tensors
    ctx, ys =  torch.tensor(xs, dtype=torch.int64), torch.tensor(ys, dtype=torch.int64)
    # encode the context vectors using one hot encoding 
    xs_enc = encode_context(ctx = ctx, num_classes=vocab.n_unique)
    # create the dataset object and return
    return Dataset(
        X = xs_enc,
        Y = ys
    )

### E0. Generalized class for dry-running bigrams and trigram models using both counting and gradient based approaches.

### E1. implementation of trigram model using the count and / or gradient based approach.

**ANS E1** : the loss for trigram is only ever so sligthly better.

In [11]:
class NGramModel:
    """Common class for a count / gradient based NGram implementation

        Object Members
            data_set:list[str] - list of sequences to model
            vocab:Vocab - object containing information about the ngram models vocabulary
            n_gram:NGram - information about the n_gram
            counts: torch.tensor - counts of the ngram occuring in all of the sequences.
            probs: torch.tensor - normalized counts (each row sums up to 1.) probability if the next occuring character given a n-1 context
            generator: torch.Generator - a generator object to be used across all the ngram impplementations for reproducible results
    
    """
    def __init__(self, data_set: list[str], n: int, config: ModelConfigs)->None:
        self.config = config
        self.data_set = data_set
        self.vocab = create_vocab(data_set=data_set, start_boundary_char=config.START_BOUNDARY_CHAR, end_boundary_char=config.END_BOUNDARY_CHAR)
        self.n_gram = generate_n_grams(data_set= data_set, start_boundary_char=config.START_BOUNDARY_CHAR,end_boundary_char=config.END_BOUNDARY_CHAR, n = n)
        self.counts = self.create_counts_array()
        self.probs = self.counts.float()
        self.probs /= self.probs.sum(dim=-1, keepdim=True)
        self.generator = torch.Generator().manual_seed(config.SEED_VAL)
        self.W = torch.randn(size=((self.n_gram.n - 1) * self.vocab.n_unique, self.vocab.n_unique), generator=self.generator, requires_grad=True)
    
    def create_counts_array(self)->torch.Tensor:
        """Common method to generate the counts array for an n-gram model
        """
        # use laplace smoothing from the start
        counts = torch.full(size=(self.vocab.n_unique, )*self.n_gram.n, fill_value=self.config.SMOOTHING_COUNT, dtype=torch.int32)
        n = self.n_gram.n

        # traverse through all the ngrams over all the sequences
        for data in self.n_gram.data:
            # get the indices
            indices = [self.vocab.stoi[ch] for ch in islice(data, n)]
            # increment by 1
            counts[*indices] += 1
        # return the counts
        return counts
    
    
    def calculate_nll_counting_method(self)->None:
        """Common method for all ngrams to calculate the nll loss. the lower the better
        """
        # keep a running sum of the individual log probs
        log_likelihood = 0
        n = self.n_gram.n
        # iterate over all the bigrams over the data set
        for data in self.n_gram.data:
            # get the int mapping of the characters
            indices = [self.vocab.stoi[ch] for ch in islice(data, n)]
            # get the probability assigned for the bigram pair
            p = self.probs[*indices]
            # get the log of the value
            log_prob = p.log()
            # accumualate the log prob sum
            log_likelihood += log_prob.item()
        # invert the sign so we can use a loss value
        neg_log_likelihood = -log_likelihood

        # normalize to have a easily interpretable value
        neg_log_likelihood /= len(self.n_gram.data)
        # display to the user
        print(f"{neg_log_likelihood=:.4f}")
    
    
    
    def sample_names(self, sample_size: int = 5, max_length: int = 5, method="count")->None:
        """Common logic to generate sequences using the ngram counting approach
            Args:
                sample_size: int - number of sequences to be generated.
                max_length: int - number of tokens in a sequence.
                method: count | gradient - which method to use 
            Returns: None, we just print the sequences one by one.
        """
        if method.casefold() == "count":
            for _ in range(sample_size):
                out = []
                context = [self.vocab.stoi[self.vocab.start_boundary_char]] * (self.n_gram.n - 1)

                while True:
                    p = self.probs[tuple(context)]
                    next_ix = torch.multinomial(p, num_samples=1, replacement=True, generator=self.generator).item()
                    context = context[1:] + [next_ix]
                    out.append(self.vocab.itos[next_ix])
                    if next_ix == self.vocab.stoi[self.vocab.end_boundary_char]:
                        break
                    if len(out) > max_length:
                        out.append(self.vocab.end_boundary_char)
                        break
                print(''.join(out))
        elif method.casefold() == "gradient":
            for _ in range(sample_size):
                out = []
                context = [self.vocab.stoi[self.vocab.start_boundary_char]] * (self.n_gram.n - 1)

                while True:
                    xenc = F.one_hot(torch.tensor(context, dtype=torch.int64), num_classes=self.vocab.n_unique)
                    xenc = xenc.view(1, -1).float()
                    logits = xenc @ self.W
                    counts = logits.exp()
                    probs = counts / counts.sum(dim=-1, keepdim=True)
                    next_ix = torch.multinomial(probs, replacement=True, num_samples=1, generator=self.generator).item()
                    out.append(self.vocab.itos[next_ix])
                    if next_ix == self.vocab.stoi[self.vocab.end_boundary_char]:
                        break
                    if len(out) > max_length:
                        out.append(self.vocab.end_boundary_char)
                        break
                    context = context[1:] + [next_ix]
                print(''.join(out))

            


            
    def create_data_set(self)->None:
        """
        Common method to create the dataset for the gradient based algorithm

            Creates:
                self.xs: torch.tensor - that contains the context (n - 1) characters
                self.ys: torch.tensor - that contians the target character that appears after the corresponding context.
        
        """
        xs, ys = [], []
        for data in self.n_gram.data:
            context = [self.vocab.stoi[ch] for ch in data[:-1]]
            next_char = self.vocab.stoi[data[-1]]
            xs.append(context)
            ys.append(next_char)
        self.xs = torch.tensor(xs, dtype=torch.int64)
        self.ys = torch.tensor(ys, dtype=torch.int64)
    
    def train_model(self)->None:
        """
        Common method that generates sets of weights to be optimized via the gradient based algorithm
        intuition for inputs to the model is : context * num_of_unique chars, num_unique_chars
        so for bigrams it will be : (2 - 1) * 27 , 27 which denots that for the set of input puts we encode it to map to a one hot encoding of 27 classees and 27 neurons predict the probabilities of next char each neuron signifying prob for one char in vocb
        similary for trigram  it will be (3 - 1) * 27 , 27 since we input two characters worth of input this would lead to having a 54,27 weight matrix corresponding to a 2 integers one hot encoded into 27 classes.
        this method then trains for NUM_EPOCHS defined in the config class passed to the NGramModel class.
            Creates:
                self.W - weights that try to provide numbers that model the probabilites measured after normalizing the counts in counting method.
        """
        for epoch in range(self.config.NUM_EPOCHS):
            # one hot encode the information
            xenc = F.one_hot(self.xs, num_classes=self.vocab.n_unique).float()
            # this flattens out the array (batch_size, context_size, unique_char_num) -> (batch_size * context_size, n_unique)
            xenc = xenc.view(len(self.xs), -1)

            logits = xenc @ self.W

            counts = logits.exp()

            probs = counts / counts.sum(dim=1, keepdim=True)

            loss = -probs[torch.arange(len(self.xs)), self.ys].log().mean()

            if epoch % 100 == 0:
                print(f"NLL={loss.item():.4f}")
            
            
            # zero out the gradients
            self.W.grad = None

            # do backprop
            loss.backward()

            # update the weights
            self.W.data += self.config.LEARNING_RATE * self.W.grad
    

In [12]:
# get the names
names = get_names(data_path='../names.txt')
names_mixed = get_names(data_path='../names_mixed.txt')


In [13]:
config = ModelConfigs()

In [14]:
# genrerate the bigram and trigram objects
bigrams = NGramModel(data_set=names, config=config, n=2)
trigrams = NGramModel(data_set=names, config=config, n=3)

In [15]:
bigrams_mixed = NGramModel(data_set=names_mixed, config=config, n = 2)
trigrams_mixed = NGramModel(data_set=names_mixed, config=config, n = 3)

In [16]:
bigrams.sample_names(), print("*"*25), bigrams_mixed.sample_names()

jigua.
sadryr.
konini.
ddaves.
man.
*************************
jicha.
sadhyr.
kanini.
dhatas.
man.


(None, None, None)

In [17]:
trigrams.sample_names(), print("*"*25), trigrams_mixed.sample_names()

an.
lena.
jacenc.
re.
wes.
*************************
an.
lana.
jacon.
re.
wasoni.


(None, None, None)

In [18]:
bigrams.sample_names(method="gradient"), print("*"*25), bigrams_mixed.sample_names(method="gradient")

rpsgqb.
n.
kdknmr.
oz.
szxjbo.
*************************
rpsgqb.
n.
kdknmr.
oz.
szxjbo.


(None, None, None)

In [19]:
trigrams.sample_names(method="gradient"), print("*"*25), trigrams_mixed.sample_names(method="gradient")

ooaanu.
mzhllv.
zziulo.
nypgbd.
oodcnm.
*************************
zpgxmz.
zon.
zziulo.
nypgbd.
oodcnm.


(None, None, None)

In [20]:
bigrams.calculate_nll_counting_method(), bigrams_mixed.calculate_nll_counting_method()

neg_log_likelihood=2.4546
neg_log_likelihood=2.3402


(None, None)

In [21]:
trigrams.calculate_nll_counting_method(), trigrams_mixed.calculate_nll_counting_method()

neg_log_likelihood=2.0931
neg_log_likelihood=1.6546


(None, None)

In [22]:
bigrams.create_data_set()
trigrams.create_data_set()

In [23]:

bigrams.train_model()

NLL=3.7590
NLL=2.4727
NLL=2.4623
NLL=2.4593
NLL=2.4578
NLL=2.4570
NLL=2.4564
NLL=2.4560
NLL=2.4558
NLL=2.4555


In [24]:

trigrams.train_model()


NLL=4.1863
NLL=2.2634
NLL=2.2484
NLL=2.2437
NLL=2.2415
NLL=2.2403
NLL=2.2395
NLL=2.2390
NLL=2.2386
NLL=2.2384


In [25]:
bigrams.sample_names(method="count"), print("*"*25), bigrams.sample_names(method="gradient")

ze.
brylon.
hiymen.
dolede.
shahar.
*************************
ziaybe.
m.
eyli.
kotall.
inaler.


(None, None, None)

In [26]:
trigrams.sample_names(method="count"), print("*"*25), trigrams.sample_names(method="gradient")

broni.
trocca.
.
ene.
ka.
*************************
rm.
lann.
arayil.
akhe.
oland.


(None, None, None)

In [27]:
trigrams.config.LEARNING_RATE = -.001
trigrams.config.NUM_EPOCHS = 1000

## E2. Split the data into  80-10-10 split

After 1000 iterations on the training set the losses are in range of each other observed below in the code executitions.

## E5. Use cross_entropy from torch.nn.functional
the results is same as our implmentation of nll loss, we should use cross_entropy more since its more numerically stable.

In [38]:
class NGramModelV2:
    """Modified version of the NgramModel defined above to just use gradient optimization.
    """
    def __init__(self, data_set: list[str], config: ModelConfigs, vocab:Vocab, n: int):
        self.data_set = data_set
        self.config = config
        self.vocab = vocab
        self.n = n
        self.W = None
    def create_datasets(self)->None:
        """For the configured split ratios create the dev eval and test splits.
            default is : 80, 10, 10 we can change this using the config object passed to the model
        """

        # dev split end index
        # n1:n2 range of val split
        # n2: test split
        n1 = int(config.DEV_SPLIT * len(self.data_set))
        n2 = int(config.TEST_EVAL_SPLIT * len(self.data_set))

        train_set = self.data_set[:n1]
        eval_set = self.data_set[n1:n2]
        test_set = self.data_set[n2:]
        
        # use the utility function to create the train, eval and test data sets
        self.train_set = build_dataset(data_set=train_set, vocab=self.vocab,n=self.n)
        self.eval_set = build_dataset(data_set=eval_set, vocab=self.vocab, n=self.n)
        self.test_set = build_dataset(data_set=test_set, vocab=self.vocab, n=self.n)

    def train_model(self)->None:
        """function that optimizes the model weights using the training set
        """
        # do this at first run.
        if not self.W:

            g = torch.Generator().manual_seed(self.config.SEED_VAL)
            self.W = torch.randn(
                size=(
                    self.vocab.n_unique * (self.n - 1),
                    self.vocab.n_unique
                ),
                generator=g,
                requires_grad=True
            )
        # for num of epochs train the model
        for i in range(self.config.NUM_EPOCHS):
            logits = self.train_set.X @ self.W
            loss = F.cross_entropy(logits, self.train_set.Y)
            self.W.grad = None
            loss.backward()
            self.W.data += self.config.LEARNING_RATE * self.W.grad
            if i % 100 == 0:
                print(f"{loss=:.4f}")

    def evaluate(self)->None:
        """Evaluate the model performance on eval and test splits"""
        # evaluate on eval split
        logits = self.eval_set.X @ self.W
        eval_loss = F.cross_entropy(logits, self.eval_set.Y)

        # evaluate test split
        logits = self.test_set.X @ self.W
        test_loss = F.cross_entropy(logits, self.test_set.Y)

        print(f"{eval_loss=:.4f}\t{test_loss=:.4f}")
    
    def sample(self, sample_size: int = 5, max_length: int = 5)->None:
        """method to sample out the names using the model weights
            Args:
                sample_size: int = number of sequences to be generated
                max_length : int = maximum length of each sequence to be generated.
        """
        g = torch.Generator().manual_seed(self.config.SEED_VAL)
        # iteratively generate each and every sample
        for _ in range(sample_size):
            # out holds all character generated at each step
            out = []
            # for a bigram we start with "." for trigram we start with ".." as context
            context = [self.vocab.stoi[self.vocab.start_boundary_char]] * (self.n - 1)


            # keep on generating till we reach the end of sequence character.
            while True:
                # encode the context
                xenc = F.one_hot(torch.tensor(context, dtype=torch.int64), num_classes=self.vocab.n_unique)
                # flatten to pass it throught hte model
                xenc = xenc.view(1, -1).float()
                # generate the log-probs
                logits = xenc @ self.W
                # get the log-probs efficiently
                probs = F.softmax(logits, dim=-1)
                # sample out the next probable character
                next_ix = torch.multinomial(probs, replacement=True, num_samples=1, generator=g).item()
                # add to the current sequence
                out.append(self.vocab.itos[next_ix])
                # if eos then break
                if next_ix == self.vocab.stoi[self.vocab.end_boundary_char]:
                    break
                # if max length reached then end the string and break
                if len(out) > max_length:
                    out.append(self.vocab.end_boundary_char)
                    break
                # move the context ahead by one character.
                context = context[1:] + [next_ix]
            # print the generated character.
            print(''.join(out))


In [39]:
# initiate the data set
data_set = names + names_mixed
shuffle(data_set)

# init the config
config = ModelConfigs()


# vocab object
vocab = create_vocab(
    start_boundary_char=config.START_BOUNDARY_CHAR,
    end_boundary_char=config.END_BOUNDARY_CHAR,
    data_set=data_set
)



In [40]:
# init the model
model = NGramModelV2(
    data_set=data_set,
    config=config,
    vocab=vocab,
    n = 3
)

In [41]:
model.create_datasets()

In [42]:
model.train_model()

loss=4.1739
loss=2.2238
loss=2.2082
loss=2.2030
loss=2.2005
loss=2.1990
loss=2.1980
loss=2.1974
loss=2.1969
loss=2.1966


In [43]:
model.evaluate()

eval_loss=2.2022	test_loss=2.1990


In [45]:
model.sample(sample_size=5, max_length=8)

aexze.
arallurai.
orik.
ah.
arinimitt.
